# Week 7 — Tuesday Lab Notebook  
## Cleaning & Preprocessing (`missing values`, `duplicates`, `type conversions`, `string cleaning`, `apply` / `map`, `categorical encoding basics`)

### Today’s Goal

By the end of today, students should be able to:

- identify dirty data in a dataset
- detect and handle missing values
- detect and remove duplicates
- convert wrong data types into correct ones
- clean messy text columns
- use `apply()` and `map()` for simple preprocessing
- perform basic categorical encoding
- prepare a small dataset so it becomes ready for analysis or machine learning


## 3-Hour Structure

**0:00–0:20** Why cleaning matters + understand messy data  
**0:20–0:55** Missing values  
**0:55–1:20** Duplicates  
**1:20–1:50** Type conversions  
**1:50–2:20** String cleaning  
**2:20–2:40** `apply()` and `map()`  
**2:40–2:55** Basic categorical encoding  
**2:55–3:00** Recap + task briefing


## Part A — Why Data Cleaning Matters

In real life, datasets are usually not perfect.

They may contain:

- blank cells
- repeated rows
- extra spaces
- wrong spellings
- numbers stored as text
- dates stored as plain strings
- category labels written in many different ways

If we use dirty data without cleaning it, our results can become wrong.

For example:

- average salary may be wrong if salary is stored as text
- counting cities may be wrong if `Lahore`, `lahore`, and ` Lahore ` are treated as different values
- machine learning models may fail if missing values are ignored

So before analysis, we first clean the data.


In [ ]:
import pandas as pd
import numpy as np

print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)


## Part B — Create a Small Messy Dataset

We will create a small dataset that intentionally contains common problems.

Students should learn better from messy examples because that is what happens in real datasets.


In [ ]:
df = pd.DataFrame({
    "student_id": [101, 102, 103, 104, 104, 105, 106, 107],
    "name": [" Ali ", "Sara", "Ahmed", "fatima", "fatima", "Usman ", "Ayesha", None],
    "city": ["Lahore", "karachi", " Lahore ", "ISLAMABAD", "ISLAMABAD", "karachi ", None, "Lahore"],
    "age": [20, 21, np.nan, 22, 22, "23", "unknown", 20],
    "marks": [85, 90, np.nan, 78, 78, "88", "91", None],
    "gender": ["M", "F", "M", "F", "F", "M", "F", "F"],
    "department": ["CS", "SE", "CS", "IT", "IT", "SE", "CS", "CS"],
    "status": ["pass", "pass", "fail", "pass", "pass", "pass", "pass", "fail"],
    "fee_paid": ["yes", "yes", "no", "yes", "yes", "yes", "Yes", "no"]
})

df


### What problems can we already see?

This dataset contains many common issues:

- `name` has extra spaces
- one `name` value is missing
- `city` has different writing styles
- `age` has missing values and text like `unknown`
- `marks` has numbers and strings mixed together
- one row looks duplicated
- `fee_paid` has inconsistent labels like `yes` and `Yes`

This is a very good practice dataset for cleaning.


## Part C — First Inspect the Dataset

Before cleaning, always inspect the data first.

Do not start changing values blindly.


In [ ]:
print(df.head())


In [ ]:
print("Shape:", df.shape)
print("Columns:", list(df.columns))


In [ ]:
df.info()


### Important idea

`info()` helps us quickly check:

- how many rows and columns exist
- which columns have missing values
- which data type each column has

This is one of the first commands students should run on any dataset.


## Part D — Understand Missing Values

A missing value means data is not available for that cell.

In Pandas, missing values usually appear as:

- `NaN`
- `None`

Missing values are very common in survey data, student data, sales data, and sensor data.


In [ ]:
df.isnull()


The result above shows `True` where a value is missing and `False` where a value is present.


In [ ]:
df.isnull().sum()


### What does this mean?

`df.isnull().sum()` counts how many missing values exist in each column.

This is one of the most important cleaning commands in Pandas.


## Part E — Check Percentage of Missing Values

Sometimes counts are not enough.

If a dataset is large, percentage helps us decide whether a column is still useful.


In [ ]:
missing_percent = (df.isnull().sum() / len(df)) * 100
missing_percent


If a column has too many missing values, we may decide to drop it.

If only a few values are missing, we may fill them.


## Part F — Fill Missing Values with a Fixed Value

This is the simplest method.

We can fill missing text values with words like `"Unknown"`  
and numeric values with a default value like `0`.

But students should remember: simple filling is easy, but it is not always the best choice.


In [ ]:
df_fill_simple = df.copy()

df_fill_simple["name"] = df_fill_simple["name"].fillna("Unknown")
df_fill_simple["city"] = df_fill_simple["city"].fillna("Unknown")
df_fill_simple["marks"] = df_fill_simple["marks"].fillna(0)

df_fill_simple


### When is this useful?

This is useful when:

- a missing text field must not stay empty
- a quick demo is needed
- zero has a meaningful interpretation

But for some columns like marks or age, filling with `0` may be misleading.


## Part G — Fill Missing Numeric Values with Mean

For numeric columns, a common basic method is to fill missing values with the mean.

Before that, we must convert dirty numeric columns into proper numeric form.


In [ ]:
df_num = df.copy()

df_num["age"] = pd.to_numeric(df_num["age"], errors="coerce")
df_num["marks"] = pd.to_numeric(df_num["marks"], errors="coerce")

df_num[["age", "marks"]]


### Why did we use `errors="coerce"`?

Because some values are not valid numbers.

For example:

- `"unknown"` in age
- text values mixed inside numeric columns

With `errors="coerce"`, invalid values become `NaN`.

That makes cleaning easier.


In [ ]:
df_num["age"] = df_num["age"].fillna(df_num["age"].mean())
df_num["marks"] = df_num["marks"].fillna(df_num["marks"].mean())

df_num[["age", "marks"]]


This method is common in beginner preprocessing pipelines.

It keeps all rows while replacing missing numeric values with average-based estimates.


## Part H — Fill Missing Values with Mode for Categories

For categorical columns, we often fill missing values with the most frequent value.

This most frequent value is called the **mode**.


In [ ]:
df_mode = df.copy()

most_common_city = df_mode["city"].mode()[0]
print("Most common city:", most_common_city)

df_mode["city"] = df_mode["city"].fillna(most_common_city)
df_mode["city"]


Mode-based filling is useful for columns such as:

- city
- department
- gender
- status


## Part I — Drop Rows with Missing Values

Sometimes we do not want to fill missing values.

Instead, we may remove incomplete rows.

This is called dropping rows.


In [ ]:
df_drop_rows = df.copy()
df_drop_rows = df_drop_rows.dropna()

df_drop_rows


In [ ]:
print("Original shape:", df.shape)
print("After dropna shape:", df_drop_rows.shape)


### Important warning

`dropna()` can remove a lot of data.

So students should check how many rows are being lost before using it.


## Part J — Drop Missing Values from Specific Columns Only

Sometimes a row is important, and we only care whether certain columns are missing.

For example, maybe `name` and `marks` are necessary, but `city` is optional.


In [ ]:
df_drop_subset = df.copy()
df_drop_subset = df_drop_subset.dropna(subset=["name", "marks"])

df_drop_subset


This method gives more control than dropping all rows with any missing value.


## Part K — Understand Duplicates

A duplicate means repeated data.

Duplicates can happen because:

- the same form was submitted twice
- data was merged incorrectly
- the same transaction was saved again
- a copy-paste mistake happened


In [ ]:
df


In [ ]:
df.duplicated()


The output shows `True` for rows that are duplicates of earlier rows.


In [ ]:
df[df.duplicated()]


## Part L — Count Duplicate Rows

Before removing duplicates, first count them.


In [ ]:
print("Number of duplicate rows:", df.duplicated().sum())


## Part M — Remove Duplicate Rows

Now we will remove repeated rows.


In [ ]:
df_no_duplicates = df.drop_duplicates()
df_no_duplicates


In [ ]:
print("Original shape:", df.shape)
print("After removing duplicates:", df_no_duplicates.shape)


### Important idea

By default, Pandas keeps the first occurrence and removes later repeated rows.


## Part N — Remove Duplicates Based on Selected Columns

Sometimes full rows are not exactly the same, but one key field is repeated.

For example, maybe `student_id` should be unique.


In [ ]:
df.drop_duplicates(subset=["student_id"])


This keeps the first occurrence of each `student_id`.


In [ ]:
df.drop_duplicates(subset=["student_id"], keep="last")


With `keep="last"`, Pandas keeps the last repeated row instead of the first.


## Part O — Understand Wrong Data Types

A data type tells us what kind of value is stored:

- integer
- float
- string
- boolean
- datetime

If a numeric column is stored as text, many calculations will fail or behave strangely.


In [ ]:
df.dtypes


Notice that `age` and `marks` are not clean numeric columns yet.

This happened because they contain mixed values.


## Part P — Convert Text to Numbers with `pd.to_numeric()`

This is safer than direct conversion because it can handle bad values.


In [ ]:
df_types = df.copy()

df_types["age"] = pd.to_numeric(df_types["age"], errors="coerce")
df_types["marks"] = pd.to_numeric(df_types["marks"], errors="coerce")

df_types.dtypes


In [ ]:
df_types[["age", "marks"]]


Now these columns are numeric, and invalid text values became `NaN`.


## Part Q — Calculate After Type Conversion

Now that `marks` is numeric, we can calculate its mean.


In [ ]:
print("Average marks:", df_types["marks"].mean())
print("Maximum marks:", df_types["marks"].max())
print("Minimum marks:", df_types["marks"].min())


Without correct data types, these calculations would be unreliable.


## Part R — Convert to Integer Carefully

Sometimes students want to convert a column into integer type.

But if the column contains missing values, plain `int` conversion may fail.


In [ ]:
age_series = pd.to_numeric(df["age"], errors="coerce")
age_series


In [ ]:
age_filled_int = age_series.fillna(age_series.mean()).round().astype(int)
age_filled_int


Here we first:

- converted invalid values to `NaN`
- filled missing values
- rounded the numbers
- converted them to integer


## Part S — Convert Strings into Datetime

Dates often come as plain text.

We should convert them into proper datetime format so we can sort, filter, and extract year or month.


In [ ]:
attendance = pd.DataFrame({
    "student": ["Ali", "Sara", "Ahmed", "Fatima"],
    "join_date": ["2026-01-10", "2026/02/15", "March 5 2026", "invalid date"]
})

attendance


In [ ]:
attendance["join_date"] = pd.to_datetime(attendance["join_date"], errors="coerce")
attendance


Again, invalid date values become `NaT`, which is like missing datetime.


In [ ]:
attendance["year"] = attendance["join_date"].dt.year
attendance["month"] = attendance["join_date"].dt.month
attendance


## Part T — Understand String Cleaning

Text columns are often messy because of:

- extra spaces
- random upper/lower case
- inconsistent spelling
- special characters

String cleaning helps make categories consistent.


In [ ]:
df_strings = df.copy()
df_strings[["name", "city", "fee_paid"]]


## Part U — Remove Extra Spaces with `.str.strip()`


In [ ]:
df_strings["name"] = df_strings["name"].str.strip()
df_strings["city"] = df_strings["city"].str.strip()

df_strings[["name", "city"]]


`.str.strip()` removes spaces from the beginning and end of text.


## Part V — Standardize Letter Case

Different letter cases create false categories.

For example:

- `Lahore`
- `lahore`
- `LAHORE`

These should usually become one clean value.


In [ ]:
df_strings["city_lower"] = df_strings["city"].str.lower()
df_strings["city_upper"] = df_strings["city"].str.upper()
df_strings["city_title"] = df_strings["city"].str.title()

df_strings[["city", "city_lower", "city_upper", "city_title"]]


For most real datasets, title case or lower case is commonly used to standardize text.


In [ ]:
df_strings["city"] = df_strings["city"].str.strip().str.title()
df_strings["name"] = df_strings["name"].str.strip().str.title()
df_strings["fee_paid"] = df_strings["fee_paid"].str.strip().str.lower()

df_strings[["name", "city", "fee_paid"]]


Now the values look much cleaner and more consistent.


## Part W — Replace Wrong or Inconsistent Labels

Sometimes we need to replace specific values manually.


In [ ]:
df_replace = df_strings.copy()

df_replace["fee_paid"] = df_replace["fee_paid"].replace({"yes": "paid", "no": "unpaid"})
df_replace["status"] = df_replace["status"].replace({"pass": "Passed", "fail": "Failed"})

df_replace[["fee_paid", "status"]]


This method is useful when labels are known and limited.


## Part X — Use `apply()` for Row-wise or Column-wise Logic

`apply()` is useful when a simple built-in method is not enough.

It lets us apply a custom function.


In [ ]:
df_apply = df_types.copy()

def classify_marks(mark):
    if pd.isna(mark):
        return "Unknown"
    elif mark >= 85:
        return "Excellent"
    elif mark >= 70:
        return "Good"
    else:
        return "Needs Improvement"

df_apply["performance"] = df_apply["marks"].apply(classify_marks)
df_apply[["marks", "performance"]]


### Why use `apply()` here?

Because we wanted custom logic:

- high marks → `Excellent`
- medium marks → `Good`
- low marks → `Needs Improvement`


## Part Y — Use `map()` for Simple Value Mapping

`map()` is best when we already know exact value replacements.


In [ ]:
df_map = df_replace.copy()

df_map["gender_full"] = df_map["gender"].map({"M": "Male", "F": "Female"})
df_map["fee_binary"] = df_map["fee_paid"].map({"paid": 1, "unpaid": 0})

df_map[["gender", "gender_full", "fee_paid", "fee_binary"]]


### Difference between `apply()` and `map()`

Use `map()` when:

- one value directly becomes another value
- the mapping is simple and fixed

Use `apply()` when:

- we need a function
- logic is conditional
- transformation is more flexible


## Part Z — Basic Categorical Encoding

Machine learning models usually need numeric inputs.

So text categories often need encoding.


### 1) Label-style encoding with `map()`

This is simple for yes/no type columns.


In [ ]:
df_encode = df_replace.copy()

df_encode["status_encoded"] = df_encode["status"].map({"Passed": 1, "Failed": 0})
df_encode["fee_paid_encoded"] = df_encode["fee_paid"].map({"paid": 1, "unpaid": 0})

df_encode[["status", "status_encoded", "fee_paid", "fee_paid_encoded"]]


### 2) One-hot encoding with `pd.get_dummies()`

This is useful when a column has multiple categories like department or city.


In [ ]:
dept_dummies = pd.get_dummies(df_replace["department"], prefix="dept")
dept_dummies


In [ ]:
df_onehot = pd.concat([df_replace, dept_dummies], axis=1)
df_onehot


Now each department became its own binary column.

For example:

- `dept_CS`
- `dept_IT`
- `dept_SE`


## Part AA — Small Real-Life Style Example 1: Clean Student Data Step by Step

Now we will combine many steps together in one mini workflow.


In [ ]:
students = df.copy()

# clean text
students["name"] = students["name"].str.strip().str.title()
students["city"] = students["city"].str.strip().str.title()
students["fee_paid"] = students["fee_paid"].str.strip().str.lower()

# convert numeric columns
students["age"] = pd.to_numeric(students["age"], errors="coerce")
students["marks"] = pd.to_numeric(students["marks"], errors="coerce")

# fill missing values
students["name"] = students["name"].fillna("Unknown")
students["city"] = students["city"].fillna("Unknown")
students["age"] = students["age"].fillna(students["age"].mean())
students["marks"] = students["marks"].fillna(students["marks"].mean())

# remove duplicates
students = students.drop_duplicates()

students


This is a simple but realistic cleaning pipeline.

Students should learn that cleaning usually happens in steps, not in one magic command.


In [ ]:
students.info()


## Part AB — Small Real-Life Style Example 2: Product Data Cleaning

Let us practice on another dataset so students do not memorize only one pattern.


In [ ]:
products = pd.DataFrame({
    "product": [" pen", "Notebook", "Pencil ", "pen", None],
    "price": ["50", "120", "30", "50", "unknown"],
    "category": ["stationery", "Stationery", "STATIONERY", "stationery", "office "],
    "in_stock": ["yes", "yes", "no", "yes", "No"]
})

products


In [ ]:
products["product"] = products["product"].str.strip().str.title()
products["category"] = products["category"].str.strip().str.title()
products["in_stock"] = products["in_stock"].str.strip().str.lower()

products["price"] = pd.to_numeric(products["price"], errors="coerce")
products["price"] = products["price"].fillna(products["price"].mean())

products = products.drop_duplicates()

products


### What did we do here?

- removed spaces
- standardized case
- converted price into numeric form
- filled missing price
- removed duplicate rows


## Part AC — Common Beginner Mistakes

Students often make these mistakes:

1. calculating mean on text columns  
2. using `astype(int)` before handling missing values  
3. removing too many rows with `dropna()`  
4. forgetting to standardize text case  
5. assuming `Lahore` and `lahore` are the same automatically  
6. using `apply()` when `map()` or string methods are enough  
7. encoding categories before cleaning the text values


## Part AD — Mini Practice in Class

Try the following small exercises before moving to the after-lab tasks:

1. count missing values in `df`  
2. convert `marks` into numeric form  
3. fill missing `marks` with mean  
4. standardize `city` into title case  
5. replace `fee_paid` with `paid` and `unpaid`  
6. encode `gender` as `Male` and `Female`


In [ ]:
# Students can write their practice answers below this cell.


## Part AE — Recap

Today we learned how to:

- inspect dirty data
- count missing values
- fill or drop missing values
- detect and remove duplicates
- convert text columns into numeric or datetime form
- clean strings using Pandas string methods
- use `apply()` and `map()`
- perform basic categorical encoding

This is one of the most important practical topics in data analysis because real data is rarely clean.


# After-Lab Tasks (10)

### Task 1
Create a DataFrame of 6 students with columns: `name`, `city`, `age`, and `marks`. Add at least 2 missing values.

### Task 2
Use `isnull().sum()` to count missing values in each column.

### Task 3
Fill missing values in the `city` column with `"Unknown"`.

### Task 4
Convert a dirty numeric column such as `marks = ["80", "90", "absent", "75"]` into proper numeric form using `pd.to_numeric()`.

### Task 5
Fill missing numeric values in `marks` using the mean.

### Task 6
Create a DataFrame with at least 2 duplicate rows and remove them.

### Task 7
Make a column of city names with inconsistent writing like `lahore`, ` Lahore `, `LAHORE`, and clean them into one standard format.

### Task 8
Use `map()` to convert `gender` values from `M/F` into `Male/Female`.

### Task 9
Use `apply()` to create a new column called `grade_label` where:
- marks 85 or above → `High`
- marks between 70 and 84 → `Medium`
- below 70 → `Low`

### Task 10
Create one-hot encoded columns for a `department` column using `pd.get_dummies()`.


# Optional Homework Challenge

Create a small messy dataset of your own with at least:

- 8 rows
- 1 duplicate row
- 2 missing values
- 1 numeric column stored as text
- 1 text column with extra spaces or inconsistent case

Then clean the dataset completely and write a short explanation of each cleaning step you performed.
